# Week 5 — Star dataset (pandas)

**Dataset:** [Kaggle Star Dataset](https://www.kaggle.com/datasets/deepu1109/star-dataset?resource=download) — saved locally as `stars.csv` in this folder.

**Run this notebook** with the working directory set to `Week 5` so `stars.csv` resolves (e.g. open the notebook from this folder in Jupyter or VS Code).

## Three analytical questions

1. **Which star properties separate star types best?** Compare temperature, luminosity, radius, and absolute magnitude across star types.

2. **Are larger stars always brighter?** Look at radius vs luminosity (and magnitude) overall and by type.

3. **Does color reliably indicate temperature and type?** Compare color labels with temperature, spectral class, and star type.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("stars.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing {DATA_PATH.resolve()}. Put stars.csv next to this notebook or cd into Week 5."
    )

STAR_TYPE_LABELS = {
    0: "Brown Dwarf",
    1: "Red Dwarf",
    2: "White Dwarf",
    3: "Main Sequence",
    4: "Supergiant",
    5: "Hypergiant",
}

df = pd.read_csv(DATA_PATH)
df["Star_type_label"] = df["Star type"].map(STAR_TYPE_LABELS)
df.head()

## Load check: shape, dtypes, missing values

Quick sanity check before trusting group averages.

In [ ]:
df.info()
df.isnull().sum()

## Category counts

See how balanced the sample is across types, colors, and spectral classes.

In [ ]:
df["Star_type_label"].value_counts()

In [ ]:
df["Star color"].value_counts()

In [ ]:
df["Spectral Class"].value_counts()

## Hot and bright subset

Rows with temperature above 10,000 K and luminosity above 100 L/☉ (approximate “upper left” of an HR-style picture).

In [ ]:
hot_bright = df[(df["Temperature (K)"] > 10000) & (df["Luminosity(L/Lo)"] > 100)]
print(f"Rows in subset: {len(hot_bright)} of {len(df)} total")
hot_bright[
    [
        "Star_type_label",
        "Temperature (K)",
        "Luminosity(L/Lo)",
        "Radius(R/Ro)",
        "Star color",
        "Spectral Class",
    ]
].head(15)

## Question 1 — Mean numeric properties by star type

Large gaps between types suggest a variable is useful for telling types apart; small gaps mean more overlap.

In [ ]:
numeric_for_group = [
    "Temperature (K)",
    "Luminosity(L/Lo)",
    "Radius(R/Ro)",
    "Absolute magnitude(Mv)",
]
grouped = (
    df.groupby("Star_type_label", observed=True)[numeric_for_group]
    .mean()
    .round(2)
)
grouped

## Question 2 — Radius vs luminosity (overall)

Pearson correlation on the full table: 1.0 would mean radius and luminosity always move together; lower values mean the link is loose or type-dependent.

In [ ]:
r = df["Radius(R/Ro)"].corr(df["Luminosity(L/Lo)"])
print(f"Pearson r (radius vs luminosity, all rows): {r:.3f}")

## Question 3 — Mean temperature by star color

If color tracked temperature perfectly, these means would sort in a clean order. Spelling variants in `Star color` often break a simple story.

In [ ]:
color_temp = (
    df.groupby("Star color", observed=True)["Temperature (K)"]
    .mean()
    .sort_values()
)
color_temp.round(0)

## Short read (same logic as `week5_stars_analysis.py`)

Numbers below are from this file only, not a full sky survey.

In [ ]:
spread_temp = grouped["Temperature (K)"].max() - grouped["Temperature (K)"].min()
spread_lum = grouped["Luminosity(L/Lo)"].max() - grouped["Luminosity(L/Lo)"].min()
print(
    f"(1) Mean temperature spans about {spread_temp:.0f} K across types; "
    f"mean luminosity spans about {spread_lum:.1f} L/Lo — luminosity moves more in absolute terms here."
)
print(
    f"(2) Overall r(radius, luminosity) = {r:.3f} — trend exists but is not a hard rule row by row."
)
print("(3) Color vs temperature: see the table above; many color strings overlap in temperature.")